# Monthly-mean movie — salinity + currents 0-7 m

One frame per month, same styling as `Seasonality.ipynb` / `1.Clusters_*.ipynb`
(gray land over the filled contours, degree-formatted ticks, discrete colorbar).

Restricted to a single year (`YEAR` below) so the result is quick to review; set
`YEAR = None` to animate the whole record.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

# There is no system ffmpeg on levante, and the PyPI `ffmpeg` package is an unrelated
# module that ships no binary. imageio-ffmpeg bundles a static one; point matplotlib's
# mp4 writer at it explicitly, since it is not on PATH.
import imageio_ffmpeg
plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

# month labels, shared by every monthly figure in this project
labs = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
        'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

YEAR = None          # None = the full 1993-2014 record (264 frames); or e.g. 1993
FPS  = 6             # frames per second

# The mp4 is the full-quality artifact. The GIF is palette-based (256 colours) and one
# frame per file, so at 264 frames it only stays a sane size at a lower dpi.
MP4_DPI = 150
GIF_DPI = 100

## Data — the same stores as `Seasonality.ipynb`

In [2]:
sal = xr.open_dataset('Sal_m_0_7m.zarr').compute()

mesh = xr.open_dataset("/work/bk1450/b383184/Amazon/Mercator/data/Hgr_cmesh.nc").isel(t=0).compute()

u = xr.open_zarr('U_m_0_7m.zarr').compute().rename({'time_counter': 'time'})
v = xr.open_zarr('V_m_0_7m.zarr').compute().rename({'time_counter': 'time'})

# The fields are already monthly means, so a frame is just one time step -- no
# groupby needed here (that is what the climatology in Seasonality.ipynb does).
if YEAR is not None:
    sal = sal.sel(time=str(YEAR))
    u   = u.sel(time=str(YEAR))
    v   = v.sel(time=str(YEAR))

# U, V live on the staggered C-grid -> average onto the T points, as in Seasonality.ipynb
U = ((u.U.shift(x=1) + u.U) * 0.5)
V = ((v.V.shift(y=1) + v.V) * 0.5)

print(f"{sal.sizes['time']} frames: "
      f"{str(sal.time.values[0])[:7]} -> {str(sal.time.values[-1])[:7]}")

12 frames: 1993-01 -> 1993-12


In [3]:
# ── Precompute everything that doesn't change between frames ───────────
lon_T = mesh["glamt"].values
lat_T = mesh["gphit"].values

s = 10                        # quiver subsampling
lon_s = lon_T[::s, ::s]
lat_s = lat_T[::s, ::s]

extent = [-80, -10, -5, 20]
proj   = ccrs.PlateCarree()

sal_f = sal.sel(lat=slice(-5, 20), lon=slice(-80, -10)).sal
U_all = U.values              # (time, y, x) -- one load, no .isel() per frame
V_all = V.values

times = sal_f.time.values

# --- plotting config, same dicts as the other notebooks ---
cmaps  = {"sal": "jet"}
labels = {"sal": "mean salinity 0-7 m"}
LEVELS = {"sal": np.arange(30, 36, 0.5)}

# contourf draws at zorder 1, so land/coastlines must sit above it
Z_DATA, Z_LAND, Z_COAST = 1, 3, 4

## The movie

`contourf` cannot be updated in place, so each frame removes the previous
collections and redraws — the figure, axes, ticks and colorbar are built once and
persist, which is what keeps the frames visually identical to the static panels.

In [ ]:
def make_movie(da, name, quiver=True, box=True, fps=FPS, save=True):
    """One frame per month. Colormap, colorbar label and discrete levels are looked
    up from the cmaps/labels/LEVELS dicts by `name`, exactly as in plot_monthly."""
    cmap, label, levels = cmaps[name], labels[name], LEVELS[name]
    cmap = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True)

    fig, ax = plt.subplots(figsize=(9, 5), subplot_kw=dict(projection=proj))

    # static decoration: drawn once, reused by every frame
    ax.add_feature(cf.LAND, facecolor="#8e9497ff", zorder=Z_LAND)
    ax.coastlines(zorder=Z_COAST)
    ax.set_extent(extent, crs=proj)
    ax.set_xticks(np.arange(extent[0], extent[1] + 1e-6, 10), crs=proj)
    ax.set_yticks(np.arange(extent[2], extent[3] + 1e-6, 5), crs=proj)
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="°"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="°"))
    ax.tick_params(labelsize=8)

    if box:
        ax.add_patch(mpatches.Rectangle(
            xy=(-65, -2), width=20, height=20,
            linewidth=1.5, edgecolor="red", facecolor="none",
            transform=proj, zorder=Z_COAST + 1,
        ))

    state = {}   # the artists that change from frame to frame

    def draw(i):
        for artist in state.values():
            artist.remove()
        state.clear()

        t  = times[i]
        dd = da.isel(time=i)

        state["pcm"] = dd.plot.contourf(
            x="lon", y="lat", ax=ax, transform=proj,
            cmap=cmap, levels=levels, norm=norm, add_colorbar=False,
            zorder=Z_DATA,
        )
        if quiver:
            state["qv"] = ax.quiver(
                lon_s, lat_s,
                U_all[i, ::s, ::s], V_all[i, ::s, ::s],
                scale=30, width=0.002, headwidth=4,
                transform=proj, color="white", alpha=0.7,
                zorder=Z_DATA + 1,
            )

        # xarray re-labels the axes on every contourf call, so blank them here rather
        # than once up front, or "lon"/"lat" reappear from frame 0 onwards.
        ax.set_xlabel("")
        ax.set_ylabel("")

        mm = int(str(t)[5:7]) - 1
        ax.set_title(f"{labs[mm]} {str(t)[:4]}", fontsize=12)
        return list(state.values())

    # the colorbar needs a mappable, so frame 0 is drawn before the animation starts
    pcm = draw(0)[0]
    cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.10,
                        ticks=levels)
    cbar.ax.set_xlabel(label, fontsize=10)

    anim = FuncAnimation(fig, draw, frames=len(times), blit=False)

    if save:
        tag  = f"_{YEAR}" if YEAR is not None else "_all"
        stem = f"figures/movie_{name}{tag}"

        # The same FuncAnimation rendered twice, so the mp4 and the GIF are
        # frame-for-frame identical and differ only in container and dpi.
        anim.save(f"{stem}.mp4", writer=FFMpegWriter(fps=fps, bitrate=4000), dpi=MP4_DPI)
        print("wrote", f"{stem}.mp4")

        anim.save(f"{stem}.gif", writer=PillowWriter(fps=fps), dpi=GIF_DPI)
        print("wrote", f"{stem}.gif")

    plt.close(fig)
    return anim

In [ ]:
anim = make_movie(sal_f, "sal", quiver=True)

# Play the mp4 straight from disk. (to_jshtml embeds every frame as base64 in the
# notebook itself -- fine for 12 frames, tens of MB of .ipynb for 264.)
tag = f"_{YEAR}" if YEAR is not None else "_all"
HTML(f'<video src="figures/movie_sal{tag}.mp4" controls loop width="800">')

### Notes

* `YEAR = 1993` (or any other year) drops back to a 12-frame test run; `None` is the
  full 1993-2014 record. Outputs are tagged accordingly (`movie_sal_all.*`).
* `quiver=False` gives salinity alone; add a variable by adding one entry to each of
  `cmaps` / `labels` / `LEVELS`, same as in the other notebooks.
* mp4 needs an ffmpeg **binary**. The PyPI package called `ffmpeg` is not it (it ships
  no binary at all) -- `imageio-ffmpeg` is what provides the static one wired up in the
  imports cell.

In [6]:
# !pip install ffmpeg

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6126 sha256=7aacccf1a706b646aa8134094c7472feb6ea379e3953fb261caf733601b7d84c
  Stored in directory: /home/b/b383184/.cache/pip/wheels/a4/04/6c/ab972358c48aedc5be02f1d28968f2baa491a2837270932043
Successfully built ffmpeg
